# 06model_selection_error_analysis 노트북 목표

1. 05번에서 저장한 후보 모델 registry를 불러와 같은 validation/test split에서 다시 평가한다.
2. validation F1-score를 기준으로 성능 기준 최종 선택 모델을 정한다.
3. 프로젝트 제안 모델(`proposed_hybrid_mlp_04`)은 성능 순위와 별개로 계속 비교한다.
4. 선택 모델과 제안 모델의 오답 유형과 메타데이터 평균을 비교해 실패 원인을 확인한다.
5. 07번 별점 정제에서 사용할 `final_selected_model.joblib`과 `final_proposed_model.joblib`을 저장한다.


## 1. 라이브러리 로드

- 저장된 모델 bundle을 불러오고, validation/test 성능과 오답 데이터를 계산하기 위한 패키지를 임포트한다.
- `joblib`은 05번에서 저장한 모델 파일을 읽고, `sklearn.metrics`는 F1, PR-AUC, precision, recall, ROC-AUC 등을 계산한다.
- 05번과 같은 split을 재현해야 후보 모델들을 같은 조건에서 비교할 수 있다.


In [1]:
import os

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split


## 2. Validation/Test 데이터와 원본 리뷰 연결

- 02번 산출물인 `csv/reviews_embeddings_extract.csv`를 불러온다.
- 04/05번과 같은 `SEED`, 같은 stratified split으로 train/validation/test를 다시 만든다.
- validation은 모델 선택 기준으로 사용하고, test는 선택 이후 참고 성능과 오답 분석에만 사용한다.
- 예측 결과를 실제 리뷰 원문과 연결하기 위해 validation/test index 기준으로 원본 리뷰 행을 복원한다.
- `rating`은 별점 정제 대상이므로 이벤트 판별 입력 feature에서 제외한다.


In [2]:
SEED = 42
LABEL_COL = 'label'
TEXT_COL = 'cleaned_review_text'
META_COLS = ['text_length', 'emoji_count', 'photo_count']
RAW_META_COLS = ['text_length', 'emoji_count', 'photo_count']

raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')

emb_cols = [f'kcbert_{i}' for i in range(768)]
meta_cols = META_COLS
hybrid_cols = emb_cols + meta_cols

X_all = raw_df[hybrid_cols].copy()
y_all = raw_df[LABEL_COL].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all,
    y_all,
    test_size=0.3,
    random_state=SEED,
    stratify=y_all,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp,
)

val_review_df = raw_df.loc[X_val.index].copy().reset_index().rename(columns={'index': 'raw_index'})
test_review_df = raw_df.loc[X_test.index].copy().reset_index().rename(columns={'index': 'raw_index'})

text_val = val_review_df[TEXT_COL].fillna('').astype(str)
text_test = test_review_df[TEXT_COL].fillna('').astype(str)

print('validation feature shape:', X_val.shape)
print('test feature shape:', X_test.shape)
print('validation review shape:', val_review_df.shape)
print('test review shape:', test_review_df.shape)
print('validation label 분포:', y_val.value_counts().sort_index().to_dict())
print('test label 분포:', y_test.value_counts().sort_index().to_dict())
print('excluded feature:', ['rating'])


validation feature shape: (1326, 771)
test feature shape: (1327, 771)
validation review shape: (1326, 783)
test review shape: (1327, 783)
validation label 분포: {0: 854, 1: 472}
test label 분포: {0: 854, 1: 473}
excluded feature: ['rating']


/var/folders/wq/j2lqqp7n33dgrwj1r4q0nl9m0000gn/T/ipykernel_67165/1751260835.py:7: DtypeWarning: Columns (0: store_name) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')


## 3. 저장된 후보 모델 로드

- 05번에서 저장한 `outputs/baseline_model_registry.csv`를 읽어 후보 모델 목록을 만든다.
- registry에는 모델 이름, feature 종류, joblib 경로, 입력 타입이 들어 있다.
- 현재 후보는 `baseline_1`, `baseline_2`, `baseline_3`, `ablation_metadata best`, `proposed_hybrid_mlp_04`이다.
- 각 joblib bundle에는 학습된 모델, 입력 feature 목록, validation에서 선택한 threshold가 저장되어 있다.


In [3]:
MODEL_REGISTRY_PATH = 'outputs/baseline_model_registry.csv'
if not os.path.exists(MODEL_REGISTRY_PATH):
    raise FileNotFoundError('outputs/baseline_model_registry.csv가 없습니다. 05번 노트북을 먼저 실행하세요.')

model_registry = pd.read_csv(MODEL_REGISTRY_PATH)
required_registry_cols = {'model', 'feature_set', 'path', 'input_type'}
missing_registry_cols = required_registry_cols - set(model_registry.columns)
if missing_registry_cols:
    raise ValueError(f'모델 registry 필수 컬럼이 없습니다: {sorted(missing_registry_cols)}')

model_specs = model_registry[['model', 'feature_set', 'path', 'input_type']].to_dict(orient='records')
missing_paths = [spec['path'] for spec in model_specs if not os.path.exists(spec['path'])]
if missing_paths:
    raise FileNotFoundError(f'모델 파일이 없습니다: {missing_paths}')

model_bundles = {spec['model']: joblib.load(spec['path']) for spec in model_specs}

loaded_models = pd.DataFrame([
    {'model': spec['model'], 'feature_set': spec['feature_set'], 'path': spec['path'], 'input_type': spec['input_type']}
    for spec in model_specs
])
display(loaded_models)


,model,feature_set,path,input_type
0,baseline_1_tfidf_random_forest,cleaned_text_tfidf,outputs/baseline_1_tfidf_random_forest_model.j...,text
1,baseline_2_metadata_only_mlp,metadata_only,outputs/baseline_2_metadata_only_mlp_model.joblib,tabular
2,baseline_3_kcbert_only_mlp,kcbert_pca_only,outputs/baseline_3_kcbert_only_mlp_model.joblib,tabular
3,ablation_metadata_text_length_emoji_count_phot...,ablation_metadata:text_length+emoji_count+phot...,outputs/ablation_metadata_best_model.joblib,tabular
4,proposed_hybrid_mlp_04,kcbert_pca+metadata,outputs/proposed_mlp_final_model.joblib,tabular


## 4. Validation/Test 성능 비교

- 모든 후보 모델을 같은 validation/test split에 적용한다.
- 텍스트 모델은 정제 리뷰 텍스트를 입력으로 사용하고, tabular 모델은 bundle에 저장된 `feature_cols`만 사용한다.
- 각 모델은 05번 또는 04번에서 저장한 threshold로 이벤트 확률을 0/1 예측으로 변환한다.
- validation 성능은 최종 모델 선택에 사용하고, test 성능은 선택 이후 참고 지표로만 확인한다.
- 최종 선택 기준은 validation F1-score 우선, PR-AUC 보조 기준이다.


In [4]:
def predict_prob(spec, bundle, split):
    model = bundle['model']

    if split == 'validation':
        text_data = text_val
        X_data = X_val
    elif split == 'test':
        text_data = text_test
        X_data = X_test
    else:
        raise ValueError(f'Unsupported split: {split}')

    if spec['input_type'] == 'text':
        return model.predict_proba(text_data)[:, 1]

    feature_cols = bundle['feature_cols']
    return model.predict_proba(X_data[feature_cols])[:, 1]


def metric_row(spec, bundle, split, y_true, prob):
    threshold = float(bundle.get('best_threshold', bundle.get('default_threshold', 0.5)))
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])

    return {
        'model': spec['model'],
        'feature_set': spec['feature_set'],
        'split': split,
        'threshold': threshold,
        'f1': float(f1_score(y_true, pred)),
        'pr_auc': float(average_precision_score(y_true, prob)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    }


model_probabilities = {'validation': {}, 'test': {}}
rows = []

for spec in model_specs:
    bundle = model_bundles[spec['model']]

    val_prob = predict_prob(spec, bundle, split='validation')
    test_prob = predict_prob(spec, bundle, split='test')

    model_probabilities['validation'][spec['model']] = val_prob
    model_probabilities['test'][spec['model']] = test_prob

    rows.append(metric_row(spec, bundle, 'validation', y_val, val_prob))
    rows.append(metric_row(spec, bundle, 'test', y_test, test_prob))

comparison_all = pd.DataFrame(rows)

validation_comparison = (
    comparison_all[comparison_all['split'] == 'validation']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
validation_comparison.insert(0, 'rank', validation_comparison.index + 1)

test_comparison = (
    comparison_all[comparison_all['split'] == 'test']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
test_comparison.insert(0, 'rank', test_comparison.index + 1)

round_cols = {
    'threshold': 4,
    'f1': 4,
    'pr_auc': 4,
    'precision': 4,
    'recall': 4,
}

display_cols = [
    'rank',
    'model',
    'feature_set',
    'threshold',
    'f1',
    'pr_auc',
    'precision',
    'recall',
    'tn',
    'fp',
    'fn',
    'tp',
]

print('Validation 검증 데이터 모델 순위')
display(validation_comparison[display_cols].round(round_cols))

print('Test 테스트 데이터 모델 순위')
display(test_comparison[display_cols].round(round_cols))

Validation 검증 데이터 모델 순위


,rank,model,feature_set,threshold,f1,pr_auc,precision,recall,tn,fp,fn,tp
0,1,baseline_2_metadata_only_mlp,metadata_only,0.5011,0.5227,0.3958,0.3900,0.7924,269,585,98,374
1,2,ablation_metadata_text_length_emoji_count_phot...,ablation_metadata:text_length+emoji_count+phot...,0.5011,0.5227,0.3958,0.3900,0.7924,269,585,98,374
2,3,proposed_hybrid_mlp_04,kcbert_pca+metadata,0.5009,0.4462,0.3850,0.3866,0.5275,459,395,223,249
3,4,baseline_3_kcbert_only_mlp,kcbert_pca_only,0.5002,0.4420,0.3679,0.3861,0.5169,466,388,228,244
4,5,baseline_1_tfidf_random_forest,cleaned_text_tfidf,0.5018,0.2511,0.4006,0.4530,0.1737,755,99,390,82


Test 테스트 데이터 모델 순위


,rank,model,feature_set,threshold,f1,pr_auc,precision,recall,tn,fp,fn,tp
0,1,baseline_2_metadata_only_mlp,metadata_only,0.5011,0.5014,0.3563,0.3713,0.7717,236,618,108,365
1,2,ablation_metadata_text_length_emoji_count_phot...,ablation_metadata:text_length+emoji_count+phot...,0.5011,0.5014,0.3563,0.3713,0.7717,236,618,108,365
2,3,baseline_3_kcbert_only_mlp,kcbert_pca_only,0.5002,0.4501,0.4095,0.4086,0.5011,511,343,236,237
3,4,proposed_hybrid_mlp_04,kcbert_pca+metadata,0.5009,0.4405,0.4029,0.3983,0.4926,502,352,240,233
4,5,baseline_1_tfidf_random_forest,cleaned_text_tfidf,0.5018,0.2866,0.4382,0.5137,0.1987,765,89,379,94


### 성능 비교표 컬럼 설명

- `rank`: validation 또는 test 내부 순위이다. validation 순위만 모델 선택에 사용한다.
- `model`: 평가한 모델 이름이다.
- `feature_set`: 모델이 사용한 입력 feature 종류이다.
- `threshold`: 이벤트 리뷰로 판단하는 확률 기준이다.
- `f1`: precision과 recall의 균형 지표이며, 최종 선택의 1순위 기준이다.
- `pr_auc`: 이벤트 리뷰 탐지 품질을 보는 보조 지표이다.
- `precision`: 이벤트 리뷰라고 예측한 것 중 실제 이벤트 리뷰의 비율이다.
- `recall`: 실제 이벤트 리뷰 중 모델이 잡아낸 비율이다.
- `tn`: 일반 리뷰를 일반 리뷰로 맞춘 개수이다.
- `fp`: 일반 리뷰를 이벤트 리뷰로 잘못 예측한 개수이다.
- `fn`: 이벤트 리뷰를 일반 리뷰로 놓친 개수이다.
- `tp`: 이벤트 리뷰를 이벤트 리뷰로 맞춘 개수이다.

## 5. 최종 모델 선택

- validation F1-score가 가장 높은 모델을 성능 기준 최종 선택 모델로 정한다.
- F1이 같은 경우 PR-AUC를 보조 기준으로 본다.
- test 성능은 기록하지만, test 결과를 보고 선택 모델을 바꾸지 않는다.
- 프로젝트 제안 모델인 `proposed_hybrid_mlp_04`는 최종 선택 모델과 별개로 계속 비교한다.


In [5]:

selected_row = validation_comparison.iloc[0].copy()
selected_model_name = selected_row['model']
selected_spec = next(spec for spec in model_specs if spec['model'] == selected_model_name)
selected_bundle = model_bundles[selected_model_name]
selected_threshold = float(selected_bundle.get('best_threshold', selected_bundle.get('default_threshold', 0.5)))
selected_test_row = test_comparison[test_comparison['model'] == selected_model_name].iloc[0].copy()

proposed_model_name = 'proposed_hybrid_mlp_04'
proposed_spec = next(spec for spec in model_specs if spec['model'] == proposed_model_name)
proposed_bundle = model_bundles[proposed_model_name]
proposed_threshold = float(proposed_bundle.get('best_threshold', proposed_bundle.get('default_threshold', 0.5)))
proposed_row = validation_comparison[validation_comparison['model'] == proposed_model_name].iloc[0].copy()
proposed_test_row = test_comparison[test_comparison['model'] == proposed_model_name].iloc[0].copy()

selection_reason = (
    '저장된 threshold를 적용했을 때 validation 이벤트 F1이 가장 높아 선택했다. '
    'test 데이터는 최종 참고 성능과 오답 분석에만 사용했다.'
)
proposed_reason = '프로젝트 제안 구조인 KcBERT PCA + metadata hybrid 모델이므로 성능 기준 선택 모델과 별도로 저장한다.'

selection_and_proposed_display = pd.DataFrame([
    {
        'role': '성능 기준 선택 모델',
        'model': selected_model_name,
        'feature_set': selected_row['feature_set'],
        'threshold': selected_threshold,
        'validation_rank': int(selected_row['rank']),
        'validation_f1': selected_row['f1'],
        'validation_pr_auc': selected_row['pr_auc'],
        'validation_precision': selected_row['precision'],
        'validation_recall': selected_row['recall'],
        'validation_fp': int(selected_row['fp']),
        'validation_fn': int(selected_row['fn']),
        'test_f1_reference': selected_test_row['f1'],
        'test_pr_auc_reference': selected_test_row['pr_auc'],
        'test_precision_reference': selected_test_row['precision'],
        'test_recall_reference': selected_test_row['recall'],
        'test_fp_reference': int(selected_test_row['fp']),
        'test_fn_reference': int(selected_test_row['fn']),
        'note': selection_reason,
    },
    {
        'role': '프로젝트 제안 모델',
        'model': proposed_model_name,
        'feature_set': proposed_row['feature_set'],
        'threshold': proposed_threshold,
        'validation_rank': int(proposed_row['rank']),
        'validation_f1': proposed_row['f1'],
        'validation_pr_auc': proposed_row['pr_auc'],
        'validation_precision': proposed_row['precision'],
        'validation_recall': proposed_row['recall'],
        'validation_fp': int(proposed_row['fp']),
        'validation_fn': int(proposed_row['fn']),
        'test_f1_reference': proposed_test_row['f1'],
        'test_pr_auc_reference': proposed_test_row['pr_auc'],
        'test_precision_reference': proposed_test_row['precision'],
        'test_recall_reference': proposed_test_row['recall'],
        'test_fp_reference': int(proposed_test_row['fp']),
        'test_fn_reference': int(proposed_test_row['fn']),
        'note': proposed_reason,
    },
]).round({
    'threshold': 4,
    'validation_f1': 4,
    'validation_pr_auc': 4,
    'validation_precision': 4,
    'validation_recall': 4,
    'test_f1_reference': 4,
    'test_pr_auc_reference': 4,
    'test_precision_reference': 4,
    'test_recall_reference': 4,
})

display(selection_and_proposed_display)


,role,model,feature_set,threshold,validation_rank,validation_f1,validation_pr_auc,validation_precision,validation_recall,validation_fp,validation_fn,test_f1_reference,test_pr_auc_reference,test_precision_reference,test_recall_reference,test_fp_reference,test_fn_reference,note
0,성능 기준 선택 모델,baseline_2_metadata_only_mlp,metadata_only,0.5011,1,0.5227,0.3958,0.3900,0.7924,585,98,0.5014,0.3563,0.3713,0.7717,618,108,저장된 threshold를 적용했을 때 validation 이벤트 F1이 가장 높아...
1,프로젝트 제안 모델,proposed_hybrid_mlp_04,kcbert_pca+metadata,0.5009,3,0.4462,0.3850,0.3866,0.5275,395,223,0.4405,0.4029,0.3983,0.4926,352,240,프로젝트 제안 구조인 KcBERT PCA + metadata hybrid 모델이므로...


## 6. 선택 모델과 제안 모델 FP/FN 오답 추출

- 성능표만으로는 모델이 어떤 방식으로 틀리는지 알 수 없다.
- 별점 정제에서는 FP와 FN의 의미가 다르기 때문에 두 오류를 나누어 확인한다.
- FP는 일반 리뷰를 이벤트 리뷰로 잘못 판단한 경우이며, 정상 리뷰까지 제거할 위험이 있다.
- FN은 이벤트 리뷰를 일반 리뷰로 놓친 경우이며, 이벤트성 별점이 정제되지 않고 남을 위험이 있다.
- 따라서 선택 모델과 제안 모델의 FP/FN을 나란히 추출해 07번 별점 정제 방식 선택에 참고한다.

In [6]:
ERROR_TYPE_ORDER = [
    'TN_일반을_일반으로_정탐',
    'FP_일반을_이벤트로_오탐',
    'FN_이벤트를_일반으로_미탐',
    'TP_이벤트를_이벤트로_정탐',
]


def make_error_detail(model_name, model_role, model_label, threshold):
    prob = model_probabilities['test'][model_name]
    pred = (prob >= threshold).astype(int)

    frame = test_review_df.copy()
    frame['model'] = model_name
    frame['model_role'] = model_role
    frame['model_label'] = model_label
    frame['threshold'] = threshold
    frame['actual_label'] = y_test.to_numpy()
    frame['pred_label'] = pred
    frame['event_prob'] = prob
    frame['error_type'] = np.select(
        [
            (frame['actual_label'] == 0) & (frame['pred_label'] == 0),
            (frame['actual_label'] == 0) & (frame['pred_label'] == 1),
            (frame['actual_label'] == 1) & (frame['pred_label'] == 0),
            (frame['actual_label'] == 1) & (frame['pred_label'] == 1),
        ],
        ERROR_TYPE_ORDER,
        default='미분류',
    )
    return frame


def summarize_error(frame):
    summary = (
        frame.groupby('error_type')
        .agg(
            count=('error_type', 'size'),
            mean_event_prob=('event_prob', 'mean'),
            mean_photo_count=('photo_count', 'mean'),
            mean_text_length=('text_length', 'mean'),
            mean_emoji_count=('emoji_count', 'mean'),
        )
        .reset_index()
    )

    summary['error_type'] = pd.Categorical(
        summary['error_type'],
        categories=ERROR_TYPE_ORDER,
        ordered=True,
    )

    summary = (
        summary
        .sort_values('error_type')
        .reset_index(drop=True)
        .round(4)
    )

    return summary[[
        'error_type',
        'count',
        'mean_event_prob',
        'mean_photo_count',
        'mean_text_length',
        'mean_emoji_count',
    ]]


selected_error_df = make_error_detail(
    selected_model_name,
    'selected',
    f'성능 우수 선택 모델 ({selected_model_name})',
    selected_threshold,
)

proposed_error_df = make_error_detail(
    proposed_model_name,
    'proposed',
    f'프로젝트 제안 모델 ({proposed_model_name})',
    proposed_threshold,
)

error_detail = pd.concat([selected_error_df, proposed_error_df], ignore_index=True)
error_df = selected_error_df.copy()

selected_error_summary = summarize_error(selected_error_df)
proposed_error_summary = summarize_error(proposed_error_df)

print('선택 모델 예측 유형 요약')
display(selected_error_summary)

print('제안 모델 예측 유형 요약')
display(proposed_error_summary)

선택 모델 예측 유형 요약


,error_type,count,mean_event_prob,mean_photo_count,mean_text_length,mean_emoji_count
0,TN_일반을_일반으로_정탐,236,0.4469,0.0508,51.3347,0.5678
1,FP_일반을_이벤트로_오탐,618,0.5216,1.2265,56.3819,0.6133
2,FN_이벤트를_일반으로_미탐,108,0.4469,0.0648,49.1019,0.5463
3,TP_이벤트를_이벤트로_정탐,365,0.5219,1.2274,48.2219,0.6110


제안 모델 예측 유형 요약


,error_type,count,mean_event_prob,mean_photo_count,mean_text_length,mean_emoji_count
0,TN_일반을_일반으로_정탐,502,0.3518,0.7530,47.6793,0.5339
1,FP_일반을_이벤트로_오탐,352,0.6187,1.1136,65.4091,0.6960
2,FN_이벤트를_일반으로_미탐,240,0.3623,0.7917,43.8875,0.6333
3,TP_이벤트를_이벤트로_정탐,233,0.6298,1.1373,53.0944,0.5579


### 결과 해석 간략

- 선택 모델은 제안 모델보다 이벤트 리뷰를 더 많이 잡아냈다. (365 vs 233)
- 선택 모델은 일반 리뷰를 이벤트 리뷰로 오탐하는 경우도 많았다. (618 vs 352)
- 제안 모델은 일반 리뷰를 일반 리뷰로 맞추는 성향이 더 강했다. (502 vs 236)
- 제안 모델은 이벤트 리뷰를 일반 리뷰로 놓치는 경우가 더 많았다. (240 vs 108)
- 두 모델 모두 사진 수, 리뷰 길이, 이모지 수 같은 메타데이터 신호에 영향을 받는 것으로 보인다.
- 별점 정제에서는 이벤트 예측 리뷰를 단순 제거하는 방식만 쓰면 정상 리뷰 손실 또는 이벤트 리뷰 잔존 문제가 생길 수 있어, 확률 기반 가중 평균 방식도 함께 비교한다.
- 선택 모델은 공격적으로 이벤트를 잡고, 제안 모델은 보수적으로 이벤트 여부를 판별한다.

## 7. 메타데이터 중요도와 행동 패턴 점검

- 메타데이터를 사용하는 모델에 대해, 각 메타데이터가 예측에 얼마나 영향을 주는지 확인한다.
- 확인 방법은 특정 컬럼 값을 무작위로 섞은 뒤 F1-score가 얼마나 떨어지는지 보는 방식이다.
- 어떤 컬럼을 섞었을 때 F1-score가 크게 떨어지면, 모델이 그 컬럼을 중요하게 사용했다고 해석한다.
- 텍스트만 사용하는 모델은 사진 수, 글자 수, 이모지 수를 입력으로 쓰지 않으므로 이 중요도 계산에서 제외한다.
- 실제 이벤트 리뷰와 일반 리뷰의 `photo_count`, `text_length`, `emoji_count` 평균도 함께 비교한다.

In [7]:
def permutation_importance_for_tabular_model(model, X, y_true, threshold, columns, repeats=10, random_state=SEED):
    rng = np.random.default_rng(random_state)
    base_prob = model.predict_proba(X)[:, 1]
    base_pred = (base_prob >= threshold).astype(int)
    base_f1 = f1_score(y_true, base_pred)

    rows = []
    for col in columns:
        drops = []
        for _ in range(repeats):
            X_perm = X.copy()
            shuffled = X_perm[col].to_numpy().copy()
            rng.shuffle(shuffled)
            X_perm[col] = shuffled
            perm_prob = model.predict_proba(X_perm)[:, 1]
            perm_pred = (perm_prob >= threshold).astype(int)
            drops.append(base_f1 - f1_score(y_true, perm_pred))

        rows.append({
            'metadata': col,
            'base_f1': float(base_f1),
            'mean_f1_drop': float(np.mean(drops)),
            'std_f1_drop': float(np.std(drops)),
            'repeats': repeats,
        })
    return pd.DataFrame(rows).sort_values('mean_f1_drop', ascending=False).reset_index(drop=True)


def metadata_importance_for_model(model_name, model_role, model_label, spec, bundle, threshold):
    feature_cols = bundle.get('feature_cols')
    if spec['input_type'] != 'tabular' or feature_cols is None:
        return pd.DataFrame()

    importance_cols = [col for col in meta_cols if col in feature_cols]
    if not importance_cols:
        return pd.DataFrame()

    importance = permutation_importance_for_tabular_model(
        bundle['model'],
        X_test[feature_cols],
        y_test,
        threshold,
        importance_cols,
        repeats=10,
    )
    importance.insert(0, 'model_label', model_label)
    importance.insert(0, 'model_role', model_role)
    importance.insert(0, 'model', model_name)
    return importance


selected_importance = metadata_importance_for_model(
    selected_model_name,
    'selected',
    f'선택 모델 ({selected_model_name})',
    selected_spec,
    selected_bundle,
    selected_threshold,
)
proposed_importance = metadata_importance_for_model(
    proposed_model_name,
    'proposed',
    f'제안 모델 ({proposed_model_name})',
    proposed_spec,
    proposed_bundle,
    proposed_threshold,
)

metadata_importance = pd.concat([selected_importance, proposed_importance], ignore_index=True)
if metadata_importance.empty:
    print('선택 모델과 제안 모델이 메타데이터 컬럼을 직접 사용하지 않아 permutation importance를 건너뜁니다.')
else:
    metadata_importance_display = (
        metadata_importance
        .sort_values(['model_role', 'mean_f1_drop'], ascending=[True, False])
        .reset_index(drop=True)
        .round({
            'base_f1': 4,
            'mean_f1_drop': 4,
            'std_f1_drop': 4,
        })
    )
    display(metadata_importance_display)

behavior_summary = (
    error_detail.groupby(['model_role', 'model', 'actual_label'])
    .agg(
        count=('actual_label', 'size'),
        mean_photo_count=('photo_count', 'mean'),
        mean_text_length=('text_length', 'mean'),
        mean_emoji_count=('emoji_count', 'mean'),
    )
    .reset_index()
    .replace({'actual_label': {0: '일반_리뷰_0', 1: '이벤트_리뷰_1'}})
    .round(4)
)

print('실제 라벨별 메타데이터 평균')
display(behavior_summary)


,model,model_role,model_label,metadata,base_f1,mean_f1_drop,std_f1_drop,repeats
0,proposed_hybrid_mlp_04,proposed,제안 모델 (proposed_hybrid_mlp_04),text_length,0.4405,0.0084,0.0040,10
1,proposed_hybrid_mlp_04,proposed,제안 모델 (proposed_hybrid_mlp_04),photo_count,0.4405,0.0061,0.0077,10
2,proposed_hybrid_mlp_04,proposed,제안 모델 (proposed_hybrid_mlp_04),emoji_count,0.4405,-0.0010,0.0028,10
3,baseline_2_metadata_only_mlp,selected,선택 모델 (baseline_2_metadata_only_mlp),photo_count,0.5014,0.0227,0.0063,10
4,baseline_2_metadata_only_mlp,selected,선택 모델 (baseline_2_metadata_only_mlp),emoji_count,0.5014,0.0001,0.0007,10
5,baseline_2_metadata_only_mlp,selected,선택 모델 (baseline_2_metadata_only_mlp),text_length,0.5014,-0.0007,0.0026,10


실제 라벨별 메타데이터 평균


,model_role,model,actual_label,count,mean_photo_count,mean_text_length,mean_emoji_count
0,proposed,proposed_hybrid_mlp_04,일반_리뷰_0,854,0.9016,54.9871,0.6007
1,proposed,proposed_hybrid_mlp_04,이벤트_리뷰_1,473,0.9619,48.4228,0.5962
2,selected,baseline_2_metadata_only_mlp,일반_리뷰_0,854,0.9016,54.9871,0.6007
3,selected,baseline_2_metadata_only_mlp,이벤트_리뷰_1,473,0.9619,48.4228,0.5962


### 결과 해석
- 이벤트 참여 리뷰는 긴 글이나 이모지 사용보다는, 사진 첨부 여부와 더 관련이 있는 행동 패턴을 보였다. 
- 다만 차이가 크지는 않아, 메타데이터만으로 이벤트 리뷰를 안정적으로 구분하기에는 한계가 있다.

## 8. KcBERT 오답과 메타데이터 보완 효과 점검

- 오답 데이터 해석 계획에 맞춰, KcBERT 임베딩만 사용한 `baseline_3_kcbert_only_mlp`의 오답을 먼저 확인한다.
- KcBERT-only 모델과 제안 모델(`KcBERT PCA + metadata`)의 오답 개수를 비교한다.
- KcBERT-only가 틀린 리뷰 중 제안 모델이 맞춘 경우와, 두 모델이 모두 틀린 경우를 나누어 본다.
- 각 케이스의 `photo_count`, `text_length`, `emoji_count` 평균을 비교해 메타데이터가 어떤 상황에서 도움이 됐는지 확인한다.
- 이 분석은 KcBERT를 단독 최종 모델로 검증하려는 목적이 아니라, KcBERT 임베딩과 메타데이터 결합이 실제로 보완 효과를 냈는지 점검하기 위한 단계이다.


In [10]:
kcbert_only_model = 'baseline_3_kcbert_only_mlp'
hybrid_model = proposed_model_name


def get_threshold(model_name):
    bundle = model_bundles[model_name]
    return float(bundle.get('best_threshold', bundle.get('default_threshold', 0.5)))


def get_model_pred(model_name):
    threshold = get_threshold(model_name)
    prob = model_probabilities['test'][model_name]
    pred = (prob >= threshold).astype(int)
    return prob, pred, threshold


kcbert_prob, kcbert_pred, kcbert_threshold = get_model_pred(kcbert_only_model)
hybrid_prob, hybrid_pred, hybrid_threshold = get_model_pred(hybrid_model)

kcbert_error_df = test_review_df.copy()
kcbert_error_df['actual_label'] = y_test.to_numpy()
kcbert_error_df['kcbert_event_prob'] = kcbert_prob
kcbert_error_df['kcbert_pred'] = kcbert_pred
kcbert_error_df['hybrid_event_prob'] = hybrid_prob
kcbert_error_df['hybrid_pred'] = hybrid_pred
kcbert_error_df['kcbert_correct'] = kcbert_error_df['kcbert_pred'] == kcbert_error_df['actual_label']
kcbert_error_df['hybrid_correct'] = kcbert_error_df['hybrid_pred'] == kcbert_error_df['actual_label']

kcbert_wrong_mask = ~kcbert_error_df['kcbert_correct']
hybrid_wrong_mask = ~kcbert_error_df['hybrid_correct']
hybrid_recovered_mask = kcbert_wrong_mask & kcbert_error_df['hybrid_correct']
hybrid_missed_mask = kcbert_wrong_mask & hybrid_wrong_mask
hybrid_regressed_mask = kcbert_error_df['kcbert_correct'] & hybrid_wrong_mask

kcbert_wrong_count = int(kcbert_wrong_mask.sum())
hybrid_wrong_count = int(hybrid_wrong_mask.sum())
recovered_count = int(hybrid_recovered_mask.sum())
missed_count = int(hybrid_missed_mask.sum())
regressed_count = int(hybrid_regressed_mask.sum())

kcbert_correct_count = int(kcbert_error_df['kcbert_correct'].sum())
hybrid_correct_count = int(kcbert_error_df['hybrid_correct'].sum())
error_reduction_count = kcbert_wrong_count - hybrid_wrong_count
error_reduction_rate = (
    error_reduction_count / kcbert_wrong_count * 100
    if kcbert_wrong_count > 0
    else 0.0
)

threshold_summary = pd.DataFrame([
    {
        '모델 구분': 'KcBERT-only',
        '모델명': kcbert_only_model,
        'threshold': kcbert_threshold,
    },
    {
        '모델 구분': '제안 모델',
        '모델명': hybrid_model,
        'threshold': hybrid_threshold,
    },
])

error_count_summary = pd.DataFrame([
    {
        '구분': 'KcBERT-only 오답 개수',
        '개수': kcbert_wrong_count,
        '설명': 'KcBERT 임베딩만 사용한 모델이 틀린 test 리뷰 수',
    },
    {
        '구분': '제안 모델 오답 개수',
        '개수': hybrid_wrong_count,
        '설명': 'KcBERT 임베딩과 메타데이터를 결합한 모델이 틀린 test 리뷰 수',
    },
    {
        '구분': 'KcBERT 오답 중 제안 모델이 보완',
        '개수': recovered_count,
        '설명': 'KcBERT-only는 틀렸지만 제안 모델은 맞춘 경우',
    },
    {
        '구분': 'KcBERT 오답 중 제안 모델도 오답',
        '개수': missed_count,
        '설명': 'KcBERT-only와 제안 모델이 모두 틀린 경우',
    },
    {
        '구분': 'KcBERT 정답 중 제안 모델이 오답',
        '개수': regressed_count,
        '설명': 'KcBERT-only는 맞췄지만 제안 모델은 틀린 경우',
    },
])

performance_change_summary = pd.DataFrame([
    {
        '비교 기준': 'KcBERT-only 대비 제안 모델',
        'KcBERT-only 오답': kcbert_wrong_count,
        '제안 모델 오답': hybrid_wrong_count,
        '오답 감소 개수': error_reduction_count,
        '오답 감소율(%)': error_reduction_rate,
        '정답 수 증감': hybrid_correct_count - kcbert_correct_count,
    }
])

CASE_ORDER = [
    'KcBERT 오답 중 제안 모델이 보완',
    'KcBERT 오답 중 제안 모델도 오답',
    'KcBERT 정답 중 제안 모델이 오답',
    '두 모델 모두 정답',
]

kcbert_error_df['비교 케이스'] = np.select(
    [
        hybrid_recovered_mask,
        hybrid_missed_mask,
        hybrid_regressed_mask,
    ],
    CASE_ORDER[:3],
    default='두 모델 모두 정답',
)

metadata_case_summary = (
    kcbert_error_df
    .groupby('비교 케이스')
    .agg(
        count=('비교 케이스', 'size'),
        mean_photo_count=('photo_count', 'mean'),
        mean_text_length=('text_length', 'mean'),
        mean_emoji_count=('emoji_count', 'mean'),
        mean_kcbert_event_prob=('kcbert_event_prob', 'mean'),
        mean_hybrid_event_prob=('hybrid_event_prob', 'mean'),
    )
    .reset_index()
)
metadata_case_summary['비교 케이스'] = pd.Categorical(
    metadata_case_summary['비교 케이스'],
    categories=CASE_ORDER,
    ordered=True,
)
metadata_case_summary = (
    metadata_case_summary
    .sort_values('비교 케이스')
    .assign(**{'비교 케이스': lambda df: df['비교 케이스'].astype(str)})
    .round(4)
    .reset_index(drop=True)
)

display(threshold_summary.round({'threshold': 4}))

print('오답 개수 비교')
display(error_count_summary)

display(performance_change_summary.round({'오답 감소율(%)': 2}))

print('케이스별 메타데이터 평균')
display(metadata_case_summary)


,모델 구분,모델명,threshold
0,KcBERT-only,baseline_3_kcbert_only_mlp,0.5002
1,제안 모델,proposed_hybrid_mlp_04,0.5009


오답 개수 비교


,구분,개수,설명
0,KcBERT-only 오답 개수,579,KcBERT 임베딩만 사용한 모델이 틀린 test 리뷰 수
1,제안 모델 오답 개수,592,KcBERT 임베딩과 메타데이터를 결합한 모델이 틀린 test 리뷰 수
2,KcBERT 오답 중 제안 모델이 보완,157,KcBERT-only는 틀렸지만 제안 모델은 맞춘 경우
3,KcBERT 오답 중 제안 모델도 오답,422,KcBERT-only와 제안 모델이 모두 틀린 경우
4,KcBERT 정답 중 제안 모델이 오답,170,KcBERT-only는 맞췄지만 제안 모델은 틀린 경우


,비교 기준,KcBERT-only 오답,제안 모델 오답,오답 감소 개수,오답 감소율(%),정답 수 증감
0,KcBERT-only 대비 제안 모델,579,592,-13,-2.25,-13


케이스별 메타데이터 평균


,비교 케이스,count,mean_photo_count,mean_text_length,mean_emoji_count,mean_kcbert_event_prob,mean_hybrid_event_prob
0,KcBERT 오답 중 제안 모델이 보완,157,0.9554,54.4650,0.5732,0.5246,0.4762
1,KcBERT 오답 중 제안 모델도 오답,422,0.9668,54.6540,0.7512,0.5182,0.5133
2,KcBERT 정답 중 제안 모델이 오답,170,1.0235,61.7235,0.4706,0.4735,0.5182
3,두 모델 모두 정답,578,0.8529,48.0190,0.5329,0.4272,0.4301


- 오히려 메타데이터가 성능 하락을 주었을 수도 있다. 특히 텍스트 길이가 큰 역할을 했을 수 도 있다.

### KcBERT의 오답/정답 샘플 저장용 코드
- 해당 코드를 통해 어떤 특징을 보일때 오답으로 판단하는가
- 은어, 신조어 등에서 어떤 한계점이 있는지 파악한다.
- KcBERT FN 샘플: 실제 이벤트인데 일반으로 예측
- KcBERT FP 샘플: 실제 일반인데 이벤트로 예측
- KcBERT TP 샘플: 실제 이벤트를 이벤트로 정답
- KcBERT TN 샘플: 실제 일반을 일반으로 정답

In [12]:
sample_cols = [
    'raw_index',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'menu_name',
    'photo_count',
    'text_length',
    'emoji_count',
    'actual_label',
    'kcbert_pred',
    'hybrid_pred',
    'kcbert_event_prob',
    'hybrid_event_prob',
    '비교 케이스',
]
sample_cols = [col for col in sample_cols if col in kcbert_error_df.columns]

kcbert_fn_samples = kcbert_error_df[
    (kcbert_error_df['actual_label'] == 1) & (kcbert_error_df['kcbert_pred'] == 0)
].sort_values('kcbert_event_prob', ascending=True)[sample_cols].head(20)

kcbert_fp_samples = kcbert_error_df[
    (kcbert_error_df['actual_label'] == 0) & (kcbert_error_df['kcbert_pred'] == 1)
].sort_values('kcbert_event_prob', ascending=False)[sample_cols].head(20)

kcbert_tp_samples = kcbert_error_df[
    (kcbert_error_df['actual_label'] == 1) & (kcbert_error_df['kcbert_pred'] == 1)
].sort_values('kcbert_event_prob', ascending=False)[sample_cols].head(20)

kcbert_tn_samples = kcbert_error_df[
    (kcbert_error_df['actual_label'] == 0) & (kcbert_error_df['kcbert_pred'] == 0)
].sort_values('kcbert_event_prob', ascending=True)[sample_cols].head(20)

kcbert_fn_samples.to_csv('outputs/kcbert_fn_samples.csv', index=False, encoding='utf-8-sig')
kcbert_fp_samples.to_csv('outputs/kcbert_fp_samples.csv', index=False, encoding='utf-8-sig')
kcbert_tp_samples.to_csv('outputs/kcbert_tp_samples.csv', index=False, encoding='utf-8-sig')
kcbert_tn_samples.to_csv('outputs/kcbert_tn_samples.csv', index=False, encoding='utf-8-sig')

## 10. 선택 모델과 제안 모델 저장

- 성능 기준 선택 모델은 `outputs/final_selected_model.joblib`로 저장한다.
- 프로젝트 제안 모델은 `outputs/final_proposed_model.joblib`로 별도 저장한다.
- 각 bundle에는 모델, 입력 타입, feature 목록, threshold, validation 선택 지표, test 참고 지표, 메타데이터 중요도 결과를 함께 넣는다.
- 이렇게 저장하면 07번에서 두 모델을 같은 방식으로 로드해 별점 정제 결과를 비교할 수 있다.


In [13]:
def make_final_bundle(bundle, model_name, row, test_row, spec, threshold, role, reason, importance):
    return {
        'model': bundle['model'],
        'model_name': model_name,
        'model_role': role,
        'feature_set': row['feature_set'],
        'input_type': spec['input_type'],
        'feature_cols': bundle.get('feature_cols'),
        'text_col': bundle.get('text_col'),
        'target_col': bundle.get('target_col', LABEL_COL),
        'default_threshold': float(bundle.get('default_threshold', 0.5)),
        'best_threshold': threshold,
        'selection_rule': 'validation F1 기준 선택 모델과 프로젝트 제안 모델을 별도 bundle로 저장',
        'selection_reason': reason,
        'validation_selected_metrics': row.to_dict(),
        'test_reference_metrics': test_row.to_dict(),
        'validation_model_comparison': validation_comparison.to_dict(orient='records'),
        'test_model_comparison': test_comparison.to_dict(orient='records'),
        'metadata_importance': importance.to_dict(orient='records'),
        'excluded_features': ['rating'],
        'next_step': '07_rating_refinement에서 이 bundle을 로드해 reviews_embeddings_extract.csv 기준 feature 행에 적용한다.',
    }


final_selected_bundle = make_final_bundle(
    selected_bundle,
    selected_model_name,
    selected_row,
    selected_test_row,
    selected_spec,
    selected_threshold,
    'selected',
    selection_reason,
    selected_importance,
)
final_proposed_bundle = make_final_bundle(
    proposed_bundle,
    proposed_model_name,
    proposed_row,
    proposed_test_row,
    proposed_spec,
    proposed_threshold,
    'proposed',
    proposed_reason,
    proposed_importance,
)

joblib.dump(final_selected_bundle, 'outputs/final_selected_model.joblib')
joblib.dump(final_proposed_bundle, 'outputs/final_proposed_model.joblib')

bundle_display_rows = []
for output_path, bundle in [
    ('outputs/final_selected_model.joblib', final_selected_bundle),
    ('outputs/final_proposed_model.joblib', final_proposed_bundle),
]:
    row = {k: v for k, v in bundle.items() if k != 'model'}
    row['output_path'] = output_path
    bundle_display_rows.append(row)

print('저장 완료: outputs/final_selected_model.joblib')
print('저장 완료: outputs/final_proposed_model.joblib')
display(pd.DataFrame(bundle_display_rows))


저장 완료: outputs/final_selected_model.joblib
저장 완료: outputs/final_proposed_model.joblib


,model_name,model_role,feature_set,input_type,feature_cols,text_col,target_col,default_threshold,best_threshold,selection_rule,selection_reason,validation_selected_metrics,test_reference_metrics,validation_model_comparison,test_model_comparison,metadata_importance,excluded_features,next_step,output_path
0,baseline_2_metadata_only_mlp,selected,metadata_only,tabular,"[text_length, emoji_count, photo_count]",None,label,0.5,0.501078,validation F1 기준 선택 모델과 프로젝트 제안 모델을 별도 bundle로 저장,저장된 threshold를 적용했을 때 validation 이벤트 F1이 가장 높아...,"{'rank': 1, 'model': 'baseline_2_metadata_only...","{'rank': 1, 'model': 'baseline_2_metadata_only...","[{'rank': 1, 'model': 'baseline_2_metadata_onl...","[{'rank': 1, 'model': 'baseline_2_metadata_onl...","[{'model': 'baseline_2_metadata_only_mlp', 'mo...",[rating],07_rating_refinement에서 이 bundle을 로드해 reviews_e...,outputs/final_selected_model.joblib
1,proposed_hybrid_mlp_04,proposed,kcbert_pca+metadata,tabular,"[kcbert_0, kcbert_1, kcbert_2, kcbert_3, kcber...",None,label,0.5,0.500871,validation F1 기준 선택 모델과 프로젝트 제안 모델을 별도 bundle로 저장,프로젝트 제안 구조인 KcBERT PCA + metadata hybrid 모델이므로...,"{'rank': 3, 'model': 'proposed_hybrid_mlp_04',...","{'rank': 4, 'model': 'proposed_hybrid_mlp_04',...","[{'rank': 1, 'model': 'baseline_2_metadata_onl...","[{'rank': 1, 'model': 'baseline_2_metadata_onl...","[{'model': 'proposed_hybrid_mlp_04', 'model_ro...",[rating],07_rating_refinement에서 이 bundle을 로드해 reviews_e...,outputs/final_proposed_model.joblib
